In [6]:
import pandas as pd
import numpy as np
import os
import re

# --- Configuration ---
PATH_ENRON_CSV = os.path.join("Raw_data", "emails.csv")
PATH_PHISHING_CSV = os.path.join("Raw_data", "phishing_emails.csv")

# --- Helper Function: Text Cleaning (Your Proven Logic) ---
def defang_text(text):
    if not isinstance(text, str):
        return ""
    
    # 1. Remove Email Headers
    if "Message-ID:" in text[:500]:
        if "\n\n" in text:
            text = text.split("\n\n", 1)[1]

    # 2. Remove HTML Tags
    text = re.sub(r'<[^>]+>', ' ', text)
    
    # 3. Remove common technical jargon/CSS
    text = re.sub(r'\b(font|face|span|style|mso|bidi|class|align)\b', ' ', text, flags=re.IGNORECASE)

    # 4. Basic cleaning
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# --- Step 1: Load Normal (Enron) Data ---
print(f"Loading Enron CSV from: {PATH_ENRON_CSV}")
try:
    df_normal = pd.read_csv(PATH_ENRON_CSV)
    
    # Robust Column Renaming
    if 'message' in df_normal.columns and 'text' not in df_normal.columns:
        df_normal.rename(columns={'message': 'text'}, inplace=True)
    elif 'content' in df_normal.columns and 'text' not in df_normal.columns:
        df_normal.rename(columns={'content': 'text'}, inplace=True)

    print("Cleaning Enron emails...")
    df_normal['text'] = df_normal['text'].astype(str).apply(defang_text)
    df_normal = df_normal[df_normal['text'].str.len() > 10].copy()
    df_normal['label'] = 0
    df_normal = df_normal[['text', 'label']]
    print(f"✅ Loaded {len(df_normal)} Normal emails.")
except FileNotFoundError:
    print(f"❌ Error: Enron file not found at {PATH_ENRON_CSV}")

# --- Step 2: Load Phishing Data (With KeyError Fix) ---
print(f"\nLoading Phishing CSV from: {PATH_PHISHING_CSV}")
try:
    df_phishing = pd.read_csv(PATH_PHISHING_CSV)
    
    # Handle Phishing-specific column names
    possible_phish_cols = ['Email Text', 'body', 'text', 'Message']
    for col in possible_phish_cols:
        if col in df_phishing.columns:
            df_phishing.rename(columns={col: 'text'}, inplace=True)
            break
            
    if 'text' not in df_phishing.columns:
        raise KeyError(f"Could not find text column. Columns are: {list(df_phishing.columns)}")

    print("Cleaning Phishing emails...")
    df_phishing['text'] = df_phishing['text'].astype(str).apply(defang_text)
    df_phishing['label'] = 1
    df_phishing = df_phishing[['text', 'label']]
    print(f"✅ Loaded {len(df_phishing)} Phishing emails.")
except Exception as e:
    print(f"❌ Error loading Phishing data: {e}")

# --- Step 3: Combine and Save ---
df_combined = pd.concat([df_normal, df_phishing], ignore_index=True)
output_path = os.path.join("Clean_data", "processed_emails.csv")
df_combined.to_csv(output_path, index=False)
print(f"\n🚀 Master Dataset saved to: {output_path}")

Loading Enron CSV from: Raw_data\emails.csv
Cleaning Enron emails...
✅ Loaded 515211 Normal emails.

Loading Phishing CSV from: Raw_data\phishing_emails.csv
Cleaning Phishing emails...
✅ Loaded 18650 Phishing emails.

🚀 Master Dataset saved to: Clean_data\processed_emails.csv


In [8]:
from sklearn.model_selection import train_test_split

# 1. LOAD THE MASTER DATASET WE JUST SAVED
df = pd.read_csv("Clean_data/processed_emails.csv")

# 2. SEPARATE THE CLASSES
safe_emails = df[df['label'] == 0]
phishing_emails = df[df['label'] == 1]

# 3. THE SECURITY RESEARCHER'S SPLIT
# Step A: Split Safe emails (80% for training/val pool, 20% for the test battlefield)
train_safe, test_safe = train_test_split(safe_emails, test_size=0.2, random_state=42)

# Step B: Take 10% of that training pool for Validation
train_safe, val_safe = train_test_split(train_safe, test_size=0.1, random_state=42)

# Step C: Build the Test Set (The Simulation)
# Combine the held-back Normal emails with ALL the Phishing emails
df_test = pd.concat([test_safe, phishing_emails]).sample(frac=1, random_state=42).reset_index(drop=True)

print("\n--- Stage 3: Final Data Distribution ---")
print(f"✅ Training Set (Normal Only): {len(train_safe)} rows")
print(f"✅ Validation Set (Normal Only): {len(val_safe)} rows")
print(f"✅ Test Set (The Simulation): {len(df_test)} rows")

# 4. SAVE AS INDIVIDUAL FILES FOR STAGE 5
train_safe.to_csv("Clean_data/train_safe.csv", index=False)
val_safe.to_csv("Clean_data/val_safe.csv", index=False)
df_test.to_csv("Clean_data/test_mixed.csv", index=False)


--- Stage 3: Final Data Distribution ---
✅ Training Set (Normal Only): 370951 rows
✅ Validation Set (Normal Only): 41217 rows
✅ Test Set (The Simulation): 121693 rows
